In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
!pip install -q peft

import pandas as pd
import torch
import numpy as np

from transformers import AutoTokenizer, AutoModelForMultipleChoice, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset

train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")

label2id = {"A": 0, "B": 1, "C": 2, "D": 3, "E": 4}
id2label = {0: "A", 1: "B", 2: "C", 3: "D", 4: "E"}

train["label"] = train["answer"].map(label2id)

train.head()

# Multiple-Choice Data Formatting
In this section, you will convert the Kaggle MCQ format into the structure required by multiple-choice models. Each question has one prompt and five options and each option must be paired with the prompt separately.
- Q1. Label Encoding
Convert the answer column in train.csv into numeric labels using the following mapping:
A = 0
B = 1
C = 2
D = 3
E = 4

- What is the encoded numeric label for the row at index 150?

In [ ]:
answer_150 = train.loc[150, "answer"]
encoded_150 = label2id[answer_150]

print("Answer at index 150:", answer_150)
print("Encoded label:", encoded_150)

# Q2. Prompt-Option Formatting
- For row index 0, create the Option B input using exactly this format:
str(prompt) + " [SEP] " + str(option_B)

- What is the exact character length of this formatted input string?

In [ ]:
formatted_B = str(train.loc[0, "prompt"]) + " [SEP] " + str(train.loc[0, "B"])

print(formatted_B)
print("Length:", len(formatted_B))

# Tokenization for Multiple-Choice Models
### Multiple-choice models expect inputs in the shape:
batch_size x num_choices x sequence_length

Since each question has five options, every row becomes five tokenized sequences.
- Q3. Single-Row MCQ Tokenization
Using bert-base-uncased, tokenize the five formatted inputs for row index 0 with:
padding = "max_length"
truncation = True
max_length = 128
return_tensors = "pt"

After reshaping for a multiple-choice model, the final input_ids tensor has shape:
[1, 5, 128]

- What is the value of the second dimension?
*


In [ ]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

row = train.loc[0]

five_inputs = [
    str(row["prompt"]) + " [SEP] " + str(row["A"]),
    str(row["prompt"]) + " [SEP] " + str(row["B"]),
    str(row["prompt"]) + " [SEP] " + str(row["C"]),
    str(row["prompt"]) + " [SEP] " + str(row["D"]),
    str(row["prompt"]) + " [SEP] " + str(row["E"]),
]

encoded = tokenizer(
    five_inputs,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

input_ids = encoded["input_ids"].unsqueeze(0)

print(input_ids.shape)
print("Second dimension:", input_ids.shape[1])

# Q4. Batch MCQ Tokenization
Tokenize the first 16 rows of train.csv as multiple-choice examples.
Each row has 5 choices.
Each choice is tokenized to length 128.

The final input_ids tensor has shape:
[16, 5, 128]

How many total token positions are in this tensor?

In [ ]:
all_inputs = []

for i in range(16):
    row = train.loc[i]
    choices = ["A", "B", "C", "D", "E"]
    
    row_inputs = [
        str(row["prompt"]) + " [SEP] " + str(row[ch])
        for ch in choices
    ]
    
    all_inputs.extend(row_inputs)

encoded_batch = tokenizer(
    all_inputs,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

input_ids_batch = encoded_batch["input_ids"].view(16, 5, 128)

print(input_ids_batch.shape)

total_positions = 16 * 5 * 128
print("Total token positions:", total_positions)

# Multiple-Choice Model Outputs
AutoModelForMultipleChoice produces one logit score for each answer option. For this competition, the model outputs five logits corresponding to A, B, C, D, and E.
- Q5. Multiple-Choice Logits
Load bert-base-uncased using AutoModelForMultipleChoice.
Tokenize row index 0 as 5 choices and pass it through the model.

The output logits tensor has shape:
[1, 5]

How many logits are produced for one question?

In [ ]:
model = AutoModelForMultipleChoice.from_pretrained("bert-base-uncased")

row = train.loc[0]

five_inputs = [
    str(row["prompt"]) + " [SEP] " + str(row["A"]),
    str(row["prompt"]) + " [SEP] " + str(row["B"]),
    str(row["prompt"]) + " [SEP] " + str(row["C"]),
    str(row["prompt"]) + " [SEP] " + str(row["D"]),
    str(row["prompt"]) + " [SEP] " + str(row["E"]),
]

encoded = tokenizer(
    five_inputs,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

mcq_inputs = {
    "input_ids": encoded["input_ids"].unsqueeze(0),
    "attention_mask": encoded["attention_mask"].unsqueeze(0)
}

with torch.no_grad():
    outputs = model(**mcq_inputs)

print(outputs.logits.shape)
print("Number of logits:", outputs.logits.shape[1])

# Q6. Supervised Loss Tensor
For row index 0, pass the tokenized 5-choice input into AutoModelForMultipleChoice along with the correct encoded label.

The model returns a scalar loss tensor.

How many dimensions does this loss tensor have?
*


In [ ]:
correct_label = torch.tensor([train.loc[0, "label"]])

with torch.no_grad():
    outputs_with_loss = model(
        input_ids=mcq_inputs["input_ids"],
        attention_mask=mcq_inputs["attention_mask"],
        labels=correct_label
    )

loss = outputs_with_loss.loss

print(loss)
print("Loss tensor dimensions:", loss.dim())

# LoRA for Efficient Fine-Tuning
- LoRA freezes most of the original model and trains only a small number of adapter parameters. This makes fine-tuning faster and more memory-efficient.
- Q7. LoRA Trainable Parameters
Apply LoRA to the bert-base-uncased multiple-choice model using:
r = 8
lora_alpha = 16
target_modules = ["query", "value"]
lora_dropout = 0.1
bias = "none"
task_type = TaskType.SEQ_CLS

Count trainable parameters using:
sum(p.numel() for p in model.parameters() if p.requires_grad)

How many parameters are trainable?
*


In [ ]:
model = AutoModelForMultipleChoice.from_pretrained("bert-base-uncased")

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS
)

lora_model = get_peft_model(model, lora_config)

trainable_params = sum(
    p.numel() for p in lora_model.parameters() if p.requires_grad
)

print("Trainable parameters:", trainable_params)

# Preparing Data for Hugging Face Trainer
- Before training, the dataset must be converted into a format that the Hugging Face Trainer can understand: tokenized input_ids, attention_mask, and numeric labels.
- Q8. Hugging Face Dataset Preparation
Create a Hugging Face Dataset from the first 100 rows of train.csv.

For each row, create:
input_ids with shape [5, 128]
attention_mask with shape [5, 128]
labels as the encoded answer label

For the first dataset item, input_ids has shape:
[5, 128]

How many tokenized choices are stored in input_ids?
*


In [ ]:
small_df = train.iloc[:100].copy()

def tokenize_mcq_example(example):
    choices = ["A", "B", "C", "D", "E"]
    
    texts = [
        str(example["prompt"]) + " [SEP] " + str(example[ch])
        for ch in choices
    ]
    
    encoded = tokenizer(
        texts,
        padding="max_length",
        truncation=True,
        max_length=128
    )
    
    return {
        "input_ids": encoded["input_ids"],
        "attention_mask": encoded["attention_mask"],
        "labels": int(example["label"])
    }

hf_dataset = Dataset.from_pandas(small_df)
tokenized_dataset = hf_dataset.map(tokenize_mcq_example)

first_item = tokenized_dataset[0]

print(np.array(first_item["input_ids"]).shape)
print("Number of tokenized choices:", len(first_item["input_ids"]))

# Tiny Fine-Tuning and Inference
In this section, you will run a very small LoRA fine-tuning job using Hugging Face Trainer. Then you will use the fine-tuned model to produce probabilities for the answer options.
- Q9. Tiny LoRA Fine-Tuning
Fine-tune a LoRA multiple-choice model on the first 32 rows using Hugging Face Trainer.

- Use the following settings:
max_length = 64
per_device_train_batch_size = 4
gradient_accumulation_steps = 1
max_steps = 4

- What is the final global_step reported by the Trainer?
*


In [ ]:
from dataclasses import dataclass
from typing import Optional, Union
from transformers.tokenization_utils_base import PreTrainedTokenizerBase, PaddingStrategy

MAX_LENGTH = 64

tiny_df = train.iloc[:32].copy()
tiny_dataset = Dataset.from_pandas(tiny_df)

def tokenize_tiny(example):
    choices = ["A", "B", "C", "D", "E"]
    
    texts = [
        str(example["prompt"]) + " [SEP] " + str(example[ch])
        for ch in choices
    ]
    
    encoded = tokenizer(
        texts,
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH
    )
    
    return {
        "input_ids": encoded["input_ids"],
        "attention_mask": encoded["attention_mask"],
        "labels": int(example["label"])
    }

tiny_tokenized = tiny_dataset.map(
    tokenize_tiny,
    remove_columns=tiny_dataset.column_names
)

@dataclass
class DataCollatorForMCQ:
    tokenizer: PreTrainedTokenizerBase
    padding: Union[bool, str, PaddingStrategy] = True
    
    def __call__(self, features):
        labels = torch.tensor([f.pop("labels") for f in features], dtype=torch.long)
        
        batch_size = len(features)
        num_choices = len(features[0]["input_ids"])
        
        flattened = []
        for feature in features:
            for i in range(num_choices):
                flattened.append({
                    "input_ids": feature["input_ids"][i],
                    "attention_mask": feature["attention_mask"][i]
                })
        
        batch = self.tokenizer.pad(
            flattened,
            padding=self.padding,
            return_tensors="pt"
        )
        
        batch = {
            k: v.view(batch_size, num_choices, -1)
            for k, v in batch.items()
        }
        
        batch["labels"] = labels
        return batch

data_collator = DataCollatorForMCQ(tokenizer)

base_model = AutoModelForMultipleChoice.from_pretrained("bert-base-uncased")

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS
)

lora_model = get_peft_model(base_model, lora_config)

training_args = TrainingArguments(
    output_dir="./tiny_lora_mcq",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    max_steps=4,
    logging_steps=1,
    save_strategy="no",
    report_to="none",
    fp16=False,
    bf16=False,
    remove_unused_columns=False
)

trainer = Trainer(
    model=lora_model,
    args=training_args,
    train_dataset=tiny_tokenized,
    data_collator=data_collator
)

train_result = trainer.train()

print("Global step:", trainer.state.global_step)

# Q10. Probability Assigned to Option E After Fine-Tuning
- Using the fine-tuned LoRA model from Q9, run inference on row index 0 and apply softmax to the logits.

- What is the probability assigned to Option E?

Round your answer to 4 decimal places.
*


In [ ]:
row = train.loc[0]

texts = [
    str(row["prompt"]) + " [SEP] " + str(row["A"]),
    str(row["prompt"]) + " [SEP] " + str(row["B"]),
    str(row["prompt"]) + " [SEP] " + str(row["C"]),
    str(row["prompt"]) + " [SEP] " + str(row["D"]),
    str(row["prompt"]) + " [SEP] " + str(row["E"]),
]

encoded = tokenizer(
    texts,
    padding="max_length",
    truncation=True,
    max_length=64,
    return_tensors="pt"
)

device = trainer.model.device

input_ids = encoded["input_ids"].unsqueeze(0).to(device)
attention_mask = encoded["attention_mask"].unsqueeze(0).to(device)

trainer.model.eval()

with torch.no_grad():
    outputs = trainer.model(
        input_ids=input_ids,
        attention_mask=attention_mask
    )

logits = outputs.logits
probs = torch.softmax(logits, dim=1)

option_e_prob = probs[0, 4].item()

print("Logits:", logits)
print("Probabilities:", probs)
print("Option E probability:", round(option_e_prob, 4))

# Milestone 5:-

# Setup (Run Before Attempting Questions) : 

- Use the following two fine-tuned sequence classification checkpoints:

- DeBERTa: microsoft/deberta-v3-small (fine-tuned checkpoint)

- RoBERTa: roberta-base (fine-tuned checkpoint)

- Label Mapping
- The models output logits for five labels corresponding to the answer options:

Label ID         Option

0                       A

1                       B

2                       C

3                       D

4                       E

Yes, I confirm that I have completed the setup successfully.
Load the fine-tuned DeBERTa and RoBERTa models.

For the prompt at row index 25, perform inference using each model independently and apply Softmax to obtain class probabilities.

- Question 1:

- Which answer option receives the highest probability from the DeBERTa model, and what is that probability?
(answer format : eg - A, probability of A)


In [3]:
!pip install -q transformers sentencepiece accelerate

import os
import numpy as np
import pandas as pd
import torch

from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 95.8 MB/s eta 0:00:00:00:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 whic

In [4]:
SEED = 42

np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", DEVICE)

Using device: cuda


In [5]:
TEST_PATH = "/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv"

# Use validation.csv if your project already has one.
# Otherwise, point this to the validation portion you created from train.csv.
VALID_PATH = "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"

DEBERTA_CHECKPOINT = "/kaggle/input/your-deberta-checkpoint"
ROBERTA_CHECKPOINT = "/kaggle/input/your-roberta-checkpoint"

OUTPUT_PATH = "/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv"

In [6]:
test_df = pd.read_csv(TEST_PATH)

print("Test shape:", test_df.shape)
print("\nColumns:")
print(test_df.columns.tolist())

display(test_df.head())

Test shape: (500, 7)

Columns:
['id', 'prompt', 'A', 'B', 'C', 'D', 'E']


,id,prompt,A,B,C,D,E
0,1,Pick the best possible answer: What is the rel...,"For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p..."
1,2,"What is the estimated redshift of CEERS-93316,...","Approximately z = 6.0, corresponding to 1 bill...","Approximately z = 16.7, corresponding to 235.8...","Approximately z = 3.0, corresponding to 5 bill...","Approximately z = 10.0, corresponding to 13 bi...","Approximately z = 13.0, corresponding to 30 bi..."
2,3,Pick the best possible answer: What is the rea...,The sun appears yellowish due to a reflection ...,"The longer wavelengths of light, such as red a...",The sun appears yellowish due to the scatterin...,The sun emits a yellow light due to its own sp...,The atmosphere absorbs the shorter wavelengths...
3,4,What is the significance of the redshift-dista...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...
4,5,What is the Landau-Lifshitz-Gilbert equation u...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...


In [7]:
OPTION_LABELS = ["A", "B", "C", "D", "E"]

def find_column(df, possible_names, required=True):
    """
    Find a dataframe column using a list of possible names.
    Matching is case-insensitive.
    """
    lower_to_original = {
        str(column).lower(): column
        for column in df.columns
    }

    for name in possible_names:
        if name.lower() in lower_to_original:
            return lower_to_original[name.lower()]

    if required:
        raise ValueError(
            f"Could not find any of these columns: {possible_names}\n"
            f"Available columns: {df.columns.tolist()}"
        )

    return None


PROMPT_COLUMN = find_column(
    test_df,
    ["prompt", "question", "text", "stem"]
)

ID_COLUMN = find_column(
    test_df,
    ["id", "index", "question_id", "row_id"],
    required=False
)

OPTION_COLUMNS = {
    "A": find_column(test_df, ["A", "option_a", "answer_a", "choice_a"], required=False),
    "B": find_column(test_df, ["B", "option_b", "answer_b", "choice_b"], required=False),
    "C": find_column(test_df, ["C", "option_c", "answer_c", "choice_c"], required=False),
    "D": find_column(test_df, ["D", "option_d", "answer_d", "choice_d"], required=False),
    "E": find_column(test_df, ["E", "option_e", "answer_e", "choice_e"], required=False),
}

print("Prompt column:", PROMPT_COLUMN)
print("ID column:", ID_COLUMN)
print("Option columns:", OPTION_COLUMNS)

Prompt column: prompt
ID column: id
Option columns: {'A': 'A', 'B': 'B', 'C': 'C', 'D': 'D', 'E': 'E'}


In [8]:
def build_input_text(row, instruction_prefix=None):
    """
    Construct the text supplied to the sequence-classification model.

    IMPORTANT:
    This format should match the format used while fine-tuning.
    """

    prompt = str(row[PROMPT_COLUMN])

    if all(OPTION_COLUMNS[label] is not None for label in OPTION_LABELS):
        choices = "\n".join(
            f"{label}. {str(row[OPTION_COLUMNS[label]])}"
            for label in OPTION_LABELS
        )

        text = f"Question: {prompt}\n\nOptions:\n{choices}"
    else:
        text = prompt

    if instruction_prefix:
        text = f"{instruction_prefix}\n{text}"

    return text


sample_text = build_input_text(test_df.iloc[25])
print(sample_text)

Question: Determine the correct option: What is the most popular explanation for the shower-curtain effect? carefully.

Options:
A. The pressure differential between the inside and outside of the shower
B. The lower in velocity resulting in an boost in pressure
C. The movement of air across the outside surface of the shower curtain
D. The use of cold water
E. Bernoulli's principle


In [9]:
DEBERTA_CHECKPOINT = "microsoft/deberta-v3-small"
ROBERTA_CHECKPOINT = "roberta-base"

In [10]:
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

def load_model_and_tokenizer(model_name_or_path):
    tokenizer = AutoTokenizer.from_pretrained(
        model_name_or_path
    )

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name_or_path,
        num_labels=5,
        ignore_mismatched_sizes=True
    )

    model.to(DEVICE)
    model.eval()

    return tokenizer, model


DEBERTA_CHECKPOINT = "microsoft/deberta-v3-small"
ROBERTA_CHECKPOINT = "roberta-base"

deberta_tokenizer, deberta_model = load_model_and_tokenizer(
    DEBERTA_CHECKPOINT
)

roberta_tokenizer, roberta_model = load_model_and_tokenizer(
    ROBERTA_CHECKPOINT
)

print("Both models loaded successfully.")
print("DeBERTa labels:", deberta_model.config.num_labels)
print("RoBERTa labels:", roberta_model.config.num_labels)

config.json:   0%|          | 0.00/578 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/286M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Both models loaded successfully.
DeBERTa labels: 5
RoBERTa labels: 5


In [11]:
@torch.inference_mode()
def predict_single_text(text, tokenizer, model, max_length=512):
    """
    Predict class probabilities for one question.
    """

    encoded = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=max_length,
        padding=True
    )

    encoded = {
        key: value.to(DEVICE)
        for key, value in encoded.items()
    }

    outputs = model(**encoded)
    logits = outputs.logits

    probabilities = torch.softmax(logits, dim=-1)

    return probabilities[0].detach().cpu().numpy()

In [12]:
ROW_INDEX = 25

row_25 = test_df.iloc[ROW_INDEX]
row_25_text = build_input_text(row_25)

deberta_prob_25 = predict_single_text(
    row_25_text,
    deberta_tokenizer,
    deberta_model
)

deberta_top_index = int(np.argmax(deberta_prob_25))
deberta_top_option = OPTION_LABELS[deberta_top_index]
deberta_top_probability = float(deberta_prob_25[deberta_top_index])

print("DeBERTa probabilities:")

for label, probability in zip(OPTION_LABELS, deberta_prob_25):
    print(f"{label}: {probability:.6f}")

print("\nQuestion 1 answer:")
print(f"{deberta_top_option}, {deberta_top_probability:.6f}")

DeBERTa probabilities:
A: 0.275879
B: 0.187988
C: 0.192017
D: 0.197021
E: 0.147095

Question 1 answer:
A, 0.275879


Using the same sample (row index 25), average the class probabilities from both models.

Average Probability = [P(DeBERTa) + P(RoBERTa)]/2

- Question 2:

Which answer option receives the highest averaged probability after simple probability ensembling?

In [13]:
roberta_prob_25 = predict_single_text(
    row_25_text,
    roberta_tokenizer,
    roberta_model
)

simple_average_25 = (
    deberta_prob_25 + roberta_prob_25
) / 2.0

simple_top_index = int(np.argmax(simple_average_25))
simple_top_option = OPTION_LABELS[simple_top_index]

print("RoBERTa probabilities:")

for label, probability in zip(OPTION_LABELS, roberta_prob_25):
    print(f"{label}: {probability:.6f}")

print("\nSimple ensemble probabilities:")

for label, probability in zip(OPTION_LABELS, simple_average_25):
    print(f"{label}: {probability:.6f}")

print("\nQuestion 2 answer:")
print(simple_top_option)

RoBERTa probabilities:
A: 0.192220
B: 0.201296
C: 0.196150
D: 0.214819
E: 0.195515

Simple ensemble probabilities:
A: 0.234050
B: 0.194642
C: 0.194083
D: 0.205920
E: 0.171305

Question 2 answer:
A


Apply weighted probability averaging after Softmax using the following weights:

DeBERTa: 0.70

RoBERTa: 0.30

Compute:

P(final) = [0.7 × P(DeBERTa)] + [0.3 × P(RoBERTa)]

# Question 3:

- Which answer option is ranked first after weighted ensembling?

In [14]:
DEBERTA_WEIGHT = 0.70
ROBERTA_WEIGHT = 0.30

weighted_prob_25 = (
    DEBERTA_WEIGHT * deberta_prob_25
    + ROBERTA_WEIGHT * roberta_prob_25
)

weighted_top_index = int(np.argmax(weighted_prob_25))
weighted_top_option = OPTION_LABELS[weighted_top_index]

print("Weighted ensemble probabilities:")

for label, probability in zip(OPTION_LABELS, weighted_prob_25):
    print(f"{label}: {probability:.6f}")

print("\nQuestion 3 answer:")
print(weighted_top_option)

Weighted ensemble probabilities:
A: 0.250781
B: 0.191981
C: 0.193244
D: 0.202385
E: 0.161621

Question 3 answer:
A


Using the weighted ensemble probabilities from Q3, rank all five answer options.

Write the final prediction exactly in Kaggle submission format.

# Question 4:

What is the Top-3 prediction string for row index 25?

*
Example : C A E



In [15]:
def probabilities_to_top3(probabilities):
    """
    Convert five class probabilities into a Kaggle Top-3 string.
    """

    ranked_indices = np.argsort(probabilities)[::-1][:3]

    ranked_labels = [
        OPTION_LABELS[index]
        for index in ranked_indices
    ]

    return " ".join(ranked_labels)


top3_row_25 = probabilities_to_top3(weighted_prob_25)

print("Question 4 answer:")
print(top3_row_25)

Question 4 answer:
A D C


Run the weighted ensemble pipeline on every row of test.csv.

Save the predictions in a file named submission.csv using the required Kaggle format:

id,prediction

where the prediction column contains the Top-3 ranked options separated by spaces.

# Question 5:

Exactly how many prediction rows are present in the generated file (excluding the header)?

In [16]:
@torch.inference_mode()
def predict_dataframe(
    dataframe,
    tokenizer,
    model,
    batch_size=8,
    instruction_prefix=None,
    max_length=512
):
    """
    Efficiently predict probabilities for every row in a dataframe.
    """

    model.eval()

    texts = [
        build_input_text(
            row,
            instruction_prefix=instruction_prefix
        )
        for _, row in dataframe.iterrows()
    ]

    all_probabilities = []

    for start in tqdm(
        range(0, len(texts), batch_size),
        desc="Running inference"
    ):
        batch_texts = texts[start:start + batch_size]

        encoded = tokenizer(
            batch_texts,
            return_tensors="pt",
            truncation=True,
            max_length=max_length,
            padding=True
        )

        encoded = {
            key: value.to(DEVICE)
            for key, value in encoded.items()
        }

        outputs = model(**encoded)

        batch_probabilities = torch.softmax(
            outputs.logits,
            dim=-1
        )

        all_probabilities.append(
            batch_probabilities.detach().cpu().numpy()
        )

    return np.concatenate(all_probabilities, axis=0)

In [17]:
deberta_test_probabilities = predict_dataframe(
    test_df,
    deberta_tokenizer,
    deberta_model,
    batch_size=8
)

roberta_test_probabilities = predict_dataframe(
    test_df,
    roberta_tokenizer,
    roberta_model,
    batch_size=8
)

weighted_test_probabilities = (
    0.70 * deberta_test_probabilities
    + 0.30 * roberta_test_probabilities
)

test_top3_predictions = [
    probabilities_to_top3(probabilities)
    for probabilities in weighted_test_probabilities
]

Running inference:   0%|          | 0/63 [00:00<?, ?it/s]

Running inference:   0%|          | 0/63 [00:00<?, ?it/s]

In [26]:
import os
import numpy as np
import pandas as pd

# Create IDs
if ID_COLUMN is not None:
    submission_ids = test_df[ID_COLUMN].values
else:
    submission_ids = np.arange(len(test_df))

# Create submission dataframe
submission_df = pd.DataFrame({
    "id": submission_ids,
    "prediction": test_top3_predictions
})

# Kaggle input is read-only, so save in working folder
OUTPUT_PATH = "/kaggle/working/submission.csv"

submission_df.to_csv(
    OUTPUT_PATH,
    index=False
)

print("Submission saved successfully:", OUTPUT_PATH)
print("Question 5 answer:", len(submission_df))

display(submission_df.head(10))

Submission saved successfully: /kaggle/working/submission.csv
Question 5 answer: 500


,id,prediction
0,1,A C B
1,2,C B A
2,3,C B A
3,4,B C A
4,5,C B A
5,6,A C D
6,7,C B A
7,8,C A B
8,9,C B A
9,10,B C A


In [27]:
prediction_row_count = len(submission_df)

print("Question 5 answer:")
print(prediction_row_count)

Question 5 answer:
500


In [28]:
submission_check = pd.read_csv(OUTPUT_PATH)

assert list(submission_check.columns) == ["id", "prediction"]
assert len(submission_check) == len(test_df)
assert submission_check["prediction"].notna().all()

print("Verified prediction rows:", len(submission_check))
print("Columns:", submission_check.columns.tolist())

Verified prediction rows: 500
Columns: ['id', 'prediction']


For the first 50 rows of test.csv, create two versions of every prompt:

1.Original prompt

2.Instruction-augmented prompt by prepending: "Answer the following multiple-choice question carefully:"

Run inference using DeBERTa on both versions.

Average the predicted probabilities from both passes.

# Question 6:

How many of the first 50 rows produce a different Top-1 prediction after applying Test-Time Augmentation?

In [29]:
TTA_PREFIX = (
    "Answer the following multiple-choice question carefully:"
)

first_50_test = test_df.iloc[:50].copy()

deberta_original_50 = predict_dataframe(
    first_50_test,
    deberta_tokenizer,
    deberta_model,
    batch_size=8,
    instruction_prefix=None
)

deberta_augmented_50 = predict_dataframe(
    first_50_test,
    deberta_tokenizer,
    deberta_model,
    batch_size=8,
    instruction_prefix=TTA_PREFIX
)

deberta_tta_50 = (
    deberta_original_50 + deberta_augmented_50
) / 2.0

Running inference:   0%|          | 0/7 [00:00<?, ?it/s]

Running inference:   0%|          | 0/7 [00:00<?, ?it/s]

In [30]:
original_top1_50 = np.argmax(
    deberta_original_50,
    axis=1
)

tta_top1_50 = np.argmax(
    deberta_tta_50,
    axis=1
)

tta_changed_count = int(
    np.sum(original_top1_50 != tta_top1_50)
)

print("Question 6 answer:")
print(tta_changed_count)

Question 6 answer:
18


Process the first 100 rows of test.csv. And compare the Top-1 prediction from:

1. DeBERTa

2. Weighted Ensemble

# Question 7:

How many rows have different Top-1 predictions?

*


In [31]:
first_100_deberta_probabilities = (
    deberta_test_probabilities[:100]
)

first_100_weighted_probabilities = (
    weighted_test_probabilities[:100]
)

deberta_top1_100 = np.argmax(
    first_100_deberta_probabilities,
    axis=1
)

ensemble_top1_100 = np.argmax(
    first_100_weighted_probabilities,
    axis=1
)

different_top1_count = int(
    np.sum(deberta_top1_100 != ensemble_top1_100)
)

print("Question 7 answer:")
print(different_top1_count)

Question 7 answer:
6


For the first 100 rows of test.csv, record the highest class probability (confidence) predicted by:

1. DeBERTa

2. Weighted Ensemble

For every row, compute:

Confidence Gain = Ensemble Confidence−DeBERTa Confidence

# Question 8:

How many rows have a positive confidence gain (greater than 0)?

In [33]:
deberta_probs
roberta_probs

NameError: name 'deberta_probs' is not defined

In [34]:
deberta_confidence_100 = np.max(
    first_100_deberta_probabilities,
    axis=1
)

ensemble_confidence_100 = np.max(
    first_100_weighted_probabilities,
    axis=1
)

confidence_gain_100 = (
    ensemble_confidence_100
    - deberta_confidence_100
)

positive_confidence_gain_count = int(
    np.sum(confidence_gain_100 > 0)
)

print("Question 8 answer:")
print(positive_confidence_gain_count)

Question 8 answer:
0


In [35]:
confidence_comparison = pd.DataFrame({
    "row_index": np.arange(100),
    "deberta_confidence": deberta_confidence_100,
    "ensemble_confidence": ensemble_confidence_100,
    "confidence_gain": confidence_gain_100,
    "positive_gain": confidence_gain_100 > 0
})

display(confidence_comparison.head(20))

/usr/local/lib/python3.12/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()


,row_index,deberta_confidence,ensemble_confidence,confidence_gain,positive_gain
0,0,0.252197,0.234145,-0.018052,False
1,1,0.260010,0.241270,-0.018740,False
2,2,0.227295,0.218435,-0.008860,False
3,3,0.255371,0.239139,-0.016232,False
4,4,0.235107,0.224944,-0.010164,False
5,5,0.240112,0.225759,-0.014353,False
6,6,0.234253,0.223699,-0.010554,False
7,7,0.258057,0.240661,-0.017396,False
8,8,0.244019,0.230842,-0.013177,False
9,9,0.224365,0.217849,-0.006516,False


For the first 100 rows of test.csv, compare the Top-3 prediction strings generated by:

1. DeBERTa alone

2. Weighted Ensemble

# Question 9:

How many rows have at least one change in their ordered Top-3 ranking after ensembling?

*
Examples:

A C D vs. A D C



In [36]:
deberta_top3_100 = [
    probabilities_to_top3(probabilities)
    for probabilities in first_100_deberta_probabilities
]

ensemble_top3_100 = [
    probabilities_to_top3(probabilities)
    for probabilities in first_100_weighted_probabilities
]

top3_changed_100 = [
    deberta_prediction != ensemble_prediction
    for deberta_prediction, ensemble_prediction
    in zip(deberta_top3_100, ensemble_top3_100)
]

different_top3_count = int(
    np.sum(top3_changed_100)
)

print("Question 9 answer:")
print(different_top3_count)

Question 9 answer:
22


In [37]:
top3_comparison = pd.DataFrame({
    "row_index": np.arange(100),
    "deberta_top3": deberta_top3_100,
    "ensemble_top3": ensemble_top3_100,
    "changed": top3_changed_100
})

display(top3_comparison[top3_comparison["changed"]].head(30))

,row_index,deberta_top3,ensemble_top3,changed
2,2,C A B,C B A,True
3,3,B A C,B C A,True
5,5,A C B,A C D,True
12,12,A B C,B A C,True
14,14,A C D,D C A,True
15,15,A C D,D C A,True
18,18,C A B,C A D,True
19,19,B A C,B C A,True
20,20,A B C,B A C,True
32,32,A C B,A C D,True


Using the Top-3 predictions generated by your weighted ensemble for the first 100 validation samples, compute the MAP@3 score.

# Question 10:


What is the final MAP@3 score? 

*
(Round to 4 decimal places.)


In [44]:
validation_df = pd.read_csv(VALID_PATH)

print("Validation shape:", validation_df.shape)
print(validation_df.columns.tolist())

display(validation_df.head())

Validation shape: (2000, 8)
['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer']


,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [45]:
ANSWER_COLUMN = find_column(
    validation_df,
    [
        "answer",
        "label",
        "target",
        "correct_answer",
        "correct_option"
    ]
)

print("Answer column:", ANSWER_COLUMN)

Answer column: answer


In [46]:
print("Prompt column:", PROMPT_COLUMN)
print("Options:", OPTION_COLUMNS)

Prompt column: prompt
Options: {'A': 'A', 'B': 'B', 'C': 'C', 'D': 'D', 'E': 'E'}


In [48]:
def normalize_answer_label(value):
    """
    Convert validation labels to A/B/C/D/E.
    Supports both letters and integer label IDs.
    """

    if pd.isna(value):
        raise ValueError("Found a missing validation label.")

    text_value = str(value).strip().upper()

    if text_value in OPTION_LABELS:
        return text_value

    # Handles strings such as "0" and "0.0"
    try:
        numeric_value = int(float(text_value))

        if 0 <= numeric_value <= 4:
            return OPTION_LABELS[numeric_value]
    except ValueError:
        pass

    raise ValueError(
        f"Unsupported answer label: {value}"
    )

In [49]:
validation_100 = validation_df.iloc[:100].copy()

deberta_validation_100 = predict_dataframe(
    validation_100,
    deberta_tokenizer,
    deberta_model,
    batch_size=16
)

roberta_validation_100 = predict_dataframe(
    validation_100,
    roberta_tokenizer,
    roberta_model,
    batch_size=16
)

weighted_validation_100 = (
    0.70 * deberta_validation_100
    + 0.30 * roberta_validation_100
)

validation_top3_100 = [
    probabilities_to_top3(probabilities)
    for probabilities in weighted_validation_100
]

true_answers_100 = [
    normalize_answer_label(answer)
    for answer in validation_100[ANSWER_COLUMN]
]

Running inference:   0%|          | 0/7 [00:00<?, ?it/s]

Running inference:   0%|          | 0/7 [00:00<?, ?it/s]

In [50]:
def apk3(actual, prediction_string):
    """
    Average Precision at 3 for one question with one correct answer.
    """

    predicted_labels = prediction_string.split()[:3]

    if actual not in predicted_labels:
        return 0.0

    rank = predicted_labels.index(actual) + 1

    return 1.0 / rank


def map_at_3(actual_answers, prediction_strings):
    """
    Mean Average Precision at 3 over all questions.
    """

    scores = [
        apk3(actual, prediction)
        for actual, prediction
        in zip(actual_answers, prediction_strings)
    ]

    return float(np.mean(scores))

In [51]:
final_map3 = map_at_3(
    true_answers_100,
    validation_top3_100
)

print("Question 10 answer:")
print(f"{final_map3:.4f}")

Question 10 answer:
0.3617


In [52]:
final_map3 = map_at_3(
    true_answers_100,
    validation_top3_100
)

print("Question 10 answer:")
print(f"{final_map3:.4f}")

Question 10 answer:
0.3617


In [53]:
validation_results = pd.DataFrame({
    "row_index": np.arange(len(validation_100)),
    "actual": true_answers_100,
    "prediction": validation_top3_100,
    "ap_at_3": [
        apk3(actual, prediction)
        for actual, prediction
        in zip(true_answers_100, validation_top3_100)
    ]
})

display(validation_results.head(20))

print(
    "Correct at rank 1:",
    np.sum(validation_results["ap_at_3"] == 1.0)
)

print(
    "Correct at rank 2:",
    np.sum(validation_results["ap_at_3"] == 0.5)
)

print(
    "Correct at rank 3:",
    np.sum(
        np.isclose(
            validation_results["ap_at_3"],
            1 / 3
        )
    )
)

print(
    "Not present in Top-3:",
    np.sum(validation_results["ap_at_3"] == 0.0)
)

,row_index,actual,prediction,ap_at_3
0,0,B,B A C,1.000000
1,1,A,C A B,0.500000
2,2,C,A D C,0.333333
3,3,B,A D B,0.333333
4,4,A,A D B,1.000000
5,5,C,A B D,0.000000
6,6,E,A C B,0.000000
7,7,A,B A C,0.500000
8,8,A,C B D,0.000000
9,9,A,C B A,0.333333


Correct at rank 1: 16
Correct at rank 2: 25
Correct at rank 3: 23
Not present in Top-3: 36
